In [1]:
import xarray as xr
import numpy as np
import glob
import warnings
warnings.filterwarnings("ignore")


def spatial_average(array, lats):
    weights = np.cos(np.deg2rad(lats))
    if len(array.shape) == 2:
        weights_array = weights[:, np.newaxis]
        weighted_array = np.multiply(array, weights_array)
        shape = np.shape(weighted_array)
        weighted_reshaped = np.reshape(weighted_array, (shape[0]*shape[1]))
        weighted_sum = np.nansum(weighted_reshaped, axis=0)
    if len(array.shape) == 3:
        weights_array = weights[np.newaxis, :, np.newaxis]
        weighted_array = np.multiply(array, weights_array)
        shape = np.shape(weighted_array)
        weighted_reshaped = np.reshape(weighted_array, (shape[0], shape[1]*shape[2]))
        weighted_sum = np.nansum(weighted_reshaped, axis=1)
    if len(array.shape) == 4:
        weights_array = weights[np.newaxis, np.newaxis, :, np.newaxis]
        weighted_array = np.multiply(array, weights_array)
        shape = np.shape(weighted_array)
        weighted_reshaped = np.reshape(weighted_array, (shape[0], shape[1], shape[2]*shape[3]))
        weighted_sum = np.nansum(weighted_reshaped, axis=2)
    if len(array.shape) == 5:
        weights_array = weights[np.newaxis, np.newaxis, np.newaxis, :, np.newaxis]
        weighted_array = np.multiply(array, weights_array)
        shape = np.shape(weighted_array)
        weighted_reshaped = np.reshape(weighted_array, (shape[0], shape[1], shape[2], shape[3]*shape[4]))
        weighted_sum = np.nansum(weighted_reshaped, axis=3)
    denom = np.nansum(weights) * 144
    return np.divide(weighted_sum, denom)

In [2]:
# Observations
paths = glob.glob('/glade/work/skygale/_processed/training-data/monthly/observations/sat/*.nc')

obs_sat = []
obs_arctic_monthly = []
obs_global_monthly = []

lats, lons = None, None

for path in paths:

    # Open dataset
    ds = xr.open_dataset(path).DATA[2:5]
    obs_sat.append(ds)

    if lats is None and lons is None:
        lats = ds.lat.values
        lons = ds.lon.values

    # Get weighted spatial average
    obs_arctic = spatial_average(np.array(ds[:, 64:]), lats[64:])
    obs_global = spatial_average(np.array(ds), lats)

    # Append for all observations
    obs_arctic_monthly.append(obs_arctic)
    obs_global_monthly.append(obs_global)

np.shape(obs_arctic_monthly)

(4, 3)

In [64]:
# Load observation based predicted internally-generated trends
arctic_obs = np.array(np.load('/glade/work/skygale/_projects/ArcticMonthlyInternal/repo_extra/CNN-output/May/Arctic_obs.npy'))[:, 0].reshape(8, 4, 3)
global_obs = np.array(np.load('/glade/work/skygale/_projects/ArcticMonthlyInternal/repo_extra/CNN-output/May/Global_obs.npy'))[:, 0]

# Remove internal variability from observations
arctic_month = np.array(obs_arctic_monthly)[:, 2]
global_month = np.array(obs_global_monthly)[:, 2]

# Store AA values by SAT dataset
AA_by_SAT = []

for SAT in range(4):

    arctic_SAT_trend = arctic_month[SAT]
    global_SAT_trend = global_month[SAT]

    AA_realizations = []

    for SLP in range(3):
        for model in range(8):

            arctic_estimate = arctic_obs[model, SAT, SLP]
            global_estimate = global_obs[model, SAT]

            arctic_ext = arctic_SAT_trend - arctic_estimate
            global_ext = global_SAT_trend - global_estimate

            AA_realizations.append(arctic_ext / global_ext)

    AA_by_SAT.append(np.array(AA_realizations))

AA_by_SAT = np.array(AA_by_SAT)  # shape = (4 SAT, 24 realizations)

AA_mean = np.nanmean(AA_by_SAT)

sigma_cnn_per_sat = np.std(AA_by_SAT, axis=1)
sigma_CNN = np.mean(sigma_cnn_per_sat)
CNN_uncertainty = 2 * sigma_CNN

AA_sat_means = np.mean(AA_by_SAT, axis=1)  # mean over CV+SLP
sigma_obs = np.std(AA_sat_means)
OBS_uncertainty = 2 * sigma_obs

sigma_total = np.sqrt(sigma_CNN**2 + sigma_obs**2)
TOTAL_uncertainty = 2 * sigma_total

print(f"Forced AA = {AA_mean:.1f}")
print(f"Observational uncertainty (2σ) = {TOTAL_uncertainty:.1f}")

Forced AA = 2.3
Observational uncertainty (2σ) = 0.2


In [18]:
# Models
paths = glob.glob('/glade/work/skygale/_processed/training-data/monthly/spliced/global/*')

model_names = []
mod_arctic_monthly_all, mod_global_monthly_all = [], []
all_members_arctic, all_members_global = [], []
all_members_arctic_sat, all_members_global_sat = [], []

for path in paths:
    model_name = path.split('/')[9][:-3]
    model_names.append(model_name)

    # Open dataset
    ds = xr.open_dataset(path)

    # Select time slice for each model
    ds_sel = ds.sel(period=1980).X_SAT_SLP[:, 2:5, :, :, 0] * (0.213/0.263)

    # Don't use Earth3 model
    if model_name == 'OthersAllEM.nc':
        ds_sel = ds_sel[7:]

    # Get weighted spatial averages
    mem_arctic_monthly = spatial_average(np.array(ds_sel[:, :, 64:]), lats[64:])
    mem_global_monthly = spatial_average(np.array(ds_sel), lats)

    # Extra test
    all_members_arctic_sat.append(mem_arctic_monthly)
    all_members_global_sat.append(mem_global_monthly)

    # Calculate ensemble means
    mod_arctic_monthly = np.nanmean(mem_arctic_monthly, axis=0)
    mod_global_monthly = np.nanmean(mem_global_monthly, axis=0)

    # Append to all models
    mod_arctic_monthly_all.append(mod_arctic_monthly)
    mod_global_monthly_all.append(mod_global_monthly)

In [60]:
# Models
months = ['March', 'April', 'May']
mod_aa70_dist, forced_AAs_LE = [], []

for i in range(3):
    print(months[i])
    for j in range(9):
        arctic_sel = all_members_arctic_sat[j]
        global_sel = all_members_global_sat[j]

        aa = np.divide(arctic_sel[:, i], global_sel[:, i])
        mod_aa70_dist.append(aa)

        LE_aa = np.nanmean(arctic_sel[:, i]) / np.mean(global_sel[:, i])
        forced_AAs_LE.append(LE_aa)

        print(f'  {model_names[j]}: {LE_aa:.1f}')

forced_AAs_LE = np.reshape(forced_AAs_LE, (9, 3))

# Rerank models in terms of AA magnitude for 70N-90N
models_reranked = ['E3SM-2-0',
                   'CanESM5',
                   'MIROC6',
                   'MPI-ESM1-2-LR',
                   'IPSL-CM6A-LR',
                   'OthersAllEM',
                   'ACCESS-ESM1-5',
                   'CESM2',
                   'CESM2_SMBB']

# Get new ranked indices
ranked_indices = []
for name in models_reranked:
    ranked_indices.append(model_names.index(name))
mod_arc70_dist = [all_members_arctic_sat[i] for i in ranked_indices]
mod_globe_dist = [all_members_global_sat[i] for i in ranked_indices]
mod_aa70_dist = [mod_aa70_dist[i] for i in ranked_indices]
new_model_names = [model_names[i] for i in ranked_indices]

# Get concatenations of distributions for normal distribution (all ensemble members together)
mod_arc70_con = np.concatenate(mod_arc70_dist, axis=0)
mod_globe_con = np.concatenate(mod_globe_dist, axis=0)
mod_aa70_con = np.concatenate(mod_aa70_dist, axis=0)

print('\nArctic:', len(mod_arc70_con), len(mod_arc70_dist))
print('Global:', len(mod_globe_con), len(mod_globe_dist))
print('AA:    ', len(mod_aa70_con), len(mod_aa70_dist))

March
  IPSL-CM6A-LR: 3.4
  CESM2_SMBB: 2.2
  CESM2: 2.3
  ACCESS-ESM1-5: 2.7
  MPI-ESM1-2-LR: 3.3
  MIROC6: 3.3
  OthersAllEM: 3.0
  E3SM-2-0: 4.7
  CanESM5: 4.0
April
  IPSL-CM6A-LR: 2.5
  CESM2_SMBB: 1.9
  CESM2: 2.0
  ACCESS-ESM1-5: 2.2
  MPI-ESM1-2-LR: 2.9
  MIROC6: 3.2
  OthersAllEM: 2.7
  E3SM-2-0: 3.8
  CanESM5: 3.2
May
  IPSL-CM6A-LR: 1.6
  CESM2_SMBB: 1.9
  CESM2: 2.0
  ACCESS-ESM1-5: 1.7
  MPI-ESM1-2-LR: 2.0
  MIROC6: 2.6
  OthersAllEM: 1.8
  E3SM-2-0: 2.4
  CanESM5: 2.1

Arctic: 321 9
Global: 321 9
AA:     321 9


In [72]:
dir_arctic = glob.glob('/glade/work/skygale/_projects/ArcticMonthlyInternal/repo_extra/CNN-output/May/arctic_*')  # <-- CHANGE
dir_global = glob.glob('/glade/work/skygale/_projects/ArcticMonthlyInternal/repo_extra/CNN-output/May/global_*')  # <-- CHANGE

# Rerank models in terms of AA magnitude
models_reranked = ['E3SM-2-0',
                   'CanESM5',
                   'MIROC6',
                   'MPI-ESM1-2-LR',
                   'IPSL-CM6A-LR',
                   'ACCESS-ESM1-5',
                   'CESM2',
                   'CESM2_SMBB']

# Arctic
arctic_internal, get_model_names = [], []
for path in dir_arctic:
    get_model_names.append(path.split('/')[9][7:-4])
    arctic_internal.append(np.load(path))

# Get new ranked indices
ranked_indices = []
for name in models_reranked:
    ranked_indices.append(get_model_names.index(name))
arctic_internal = [arctic_internal[i] for i in ranked_indices]

# Global
global_internal, get_model_names = [], []
for path in dir_global:
    get_model_names.append(path.split('/')[9][7:-4])
    global_internal.append(np.load(path))

# Get new ranked indices
ranked_indices = []
for name in models_reranked:
    ranked_indices.append(get_model_names.index(name))
global_internal = [global_internal[i] for i in ranked_indices]
get_model_names = [get_model_names[i] for i in ranked_indices]

print('Models\nArctic:')
for i in range(len(arctic_internal)):
    print(i+1, arctic_internal[i].shape)
print('\nGlobal:')
for i in range(len(global_internal)):
    print(i+1, global_internal[i].shape)

# Remove OthersAllEM
print(f'\n{new_model_names}')

remove = 5
mod_arc70_dist_ = mod_arc70_dist[:remove] + mod_arc70_dist[remove+1:]
mod_globe_dist_ = mod_globe_dist[:remove] + mod_globe_dist[remove+1:]
mod_aa70_dist_ = mod_aa70_dist[:remove] + mod_aa70_dist[remove+1:]
new_model_names_ = new_model_names[:remove] + new_model_names[remove+1:]

print(new_model_names_)

Models
Arctic:
1 (2, 21, 3)
2 (2, 25, 3)
3 (2, 50, 3)
4 (2, 10, 3)
5 (2, 11, 3)
6 (2, 25, 3)
7 (2, 50, 3)
8 (2, 50, 3)

Global:
1 (2, 21, 3)
2 (2, 25, 3)
3 (2, 50, 3)
4 (2, 10, 3)
5 (2, 11, 3)
6 (2, 25, 3)
7 (2, 50, 3)
8 (2, 50, 3)

['E3SM-2-0', 'CanESM5', 'MIROC6', 'MPI-ESM1-2-LR', 'IPSL-CM6A-LR', 'OthersAllEM', 'ACCESS-ESM1-5', 'CESM2', 'CESM2_SMBB']
['E3SM-2-0', 'CanESM5', 'MIROC6', 'MPI-ESM1-2-LR', 'IPSL-CM6A-LR', 'ACCESS-ESM1-5', 'CESM2', 'CESM2_SMBB']


In [73]:
# Remove internal variability: Models
mod_arc70_dist_rem = []
mod_globe_dist_rem = []
mod_aa70_dist_rem = []

for i in range(8):
    mod_arc70_dist_rem.append(mod_arc70_dist_[i][:, 2] - arctic_internal[i][0, :, 0])  # <-- CHANGE dist
    mod_globe_dist_rem.append(mod_globe_dist_[i][:, 2] - global_internal[i][0, :, 0])  # <-- CHANGE dist
    mod_aa70_dist_rem.append(np.divide(mod_arc70_dist_rem[i], mod_globe_dist_rem[i]))

# Concatentate distributions
mod_arc70_con_rem = np.concatenate(mod_arc70_dist_rem, axis=0)
mod_globe_con_rem = np.concatenate(mod_globe_dist_rem, axis=0)
mod_aa70_con_rem = np.concatenate(mod_aa70_dist_rem, axis=0)

# Check shapes
for i in range(len(mod_aa70_dist_rem)):
    print(i+1, mod_aa70_dist_rem[i].shape)

print('\nTotal:', mod_aa70_con_rem.shape)

1 (21,)
2 (25,)
3 (50,)
4 (10,)
5 (11,)
6 (25,)
7 (50,)
8 (50,)

Total: (242,)


In [74]:
# Models overall CNN uncertainty
forced_AAs_LE_month = forced_AAs_LE[:, 2]          # <-- CHANGE
arctic_month = np.array(obs_arctic_monthly)[:, 2]  # <-- CHANGE
global_month = np.array(obs_global_monthly)[:, 2]  # <-- CHANGE

AA_error_distribution = []
for i in range(8):
    AA_error_distribution.append(mod_aa70_dist_rem[i] - forced_AAs_LE_month[i])

new_errors = []
for error in AA_error_distribution:
    squared_error = np.multiply(error, error)
    error_new = squared_error / len(squared_error)
    new_errors.append(error_new)

new_errors_all = np.concatenate(new_errors, axis=0)
new_errors_sum = np.nansum(new_errors_all, axis=0)
new_errors_mean = new_errors_sum / 8
new_errors_std = np.sqrt(new_errors_mean)

errors_concat = np.concatenate(AA_error_distribution, axis=0)

# Observations sample spread uncertainty
rem_AA_spread = []
for SAT in range(4):
    arctic_SAT_trend = arctic_month[SAT]
    global_SAT_trend = global_month[SAT]

    for model in range(8):
        global_estimate = global_obs[model, SAT]
        global_rem_num = global_SAT_trend - global_estimate

        for SLP in range(3):
            arctic_estimate = arctic_obs[model, SAT, SLP]
            arctic_rem_num = arctic_SAT_trend - arctic_estimate
            rem_AA_spread.append(arctic_rem_num / global_rem_num)

rem_AA_spread = np.reshape(rem_AA_spread, (4, 8, 3))
rem_AA_spread = np.mean(rem_AA_spread, axis=1)

# Observations overall uncertainty
sigma_obs = np.nanstd(np.ravel(rem_AA_spread))

new_sigma_squared = sigma_obs**2 + new_errors_std**2
new_sigma = np.sqrt(new_sigma_squared)

print('mean :', np.nanmean(errors_concat))
print('sigma:', np.std(errors_concat))
print('2 sigma obs:', 2*sigma_obs)
print('2 sigma cnn:', 2*new_errors_std)
print('sigma new:', new_sigma)

mean : -0.851560664636387
sigma: 0.6876291894758871
2 sigma obs: 0.1490957551554172
2 sigma cnn: 2.2249652552867922
sigma new: 1.1149775705635052
